# exp_nqs_mechanism_diagnose — what is actually burying rel=2 eligibles in the NQS ensemble?

CPU-only (all NQS CE scores cached). No GPU needed.

Reloads NQS feature caches, re-fits the multi-view LambdaMART (seconds), then runs:
1. **Feature profile by outcome** — surfaced_r2 vs buried_r2 vs FP_top10
2. **Displacer analysis** — which feature makes the top-10 FPs beat the buried eligibles?
3. **Eligibility retention** — does elig_first-L512 already cover the buried docs' eligibility text?
   (The key question: does clf_long add new signal, or is truncation not the mechanism?)

**Why this matters:** the old §11c/error_analysis was done with `llm_yesno` dominant. The NQS ensemble
dropped it. The burial mechanism may be completely different — this diagnostic gates the clf_long-vs-monoT5
decision before spending a GPU run on the wrong lever.

## Setup (CPU is fine — all NQS scores are cached)

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q lightgbm pytrec_eval datasets transformers pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd, lightgbm as lgb
from ctmatch.experiments import (ExperimentConfig, load_corpus, load_eval,
                                 ndcg_at_k, pytrec_metrics,
                                 eligibility_retained_frac, exclusion_retained_frac)

POOL_TAG = 'nqs'
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag=POOL_TAG)

## Load feature files (no GPU)

In [ ]:
corpus_ids, corpus_fields = load_corpus(cfg)
id2fields = dict(zip(corpus_ids, corpus_fields))
sets = load_eval(cfg, ['trec21', 'kz', 'trec22'])
pool = json.load(open(cfg.pool_path()))

rfeat = {}
for l in open(cfg.feat_file('retrieval_feats')):
    r = json.loads(l); rfeat[(r['source'], r['topic_id'], r['doc_id'])] = r

llm = {}
for l in open(cfg.feat_file('llm_scores')):
    r = json.loads(l); llm[(r['source'], r['topic_id'], r['doc_id'])] = r['llm_score']

TOPI_PATH = cfg.feat_file('topicality'); topi = {}
if os.path.exists(TOPI_PATH):
    for l in open(TOPI_PATH):
        r = json.loads(l); topi[(r['source'], r['topic_id'], r['doc_id'])] = r['topicality']
print('topicality judge:', bool(topi))

CM_PATH = cfg.feat_file('condition_match'); cm = {}
if os.path.exists(CM_PATH):
    for l in open(CM_PATH):
        r = json.loads(l); cm[(r['source'], r['topic_id'], r['doc_id'])] = r['condition_match']
print('condition_match:', bool(cm))

In [ ]:
# CE feature caches — no GPU. Will error if NQS caches don't exist yet.
clf_path  = cfg.ce_cache_path('clf_R')
clft_path = cfg.ce_cache_path('clf_topic')

if not os.path.exists(clf_path):
    raise FileNotFoundError(f'clf_R cache missing: {clf_path}\nRun train_ensemble_full with POOL_TAG="nqs" first (GPU).')

clf_f  = np.load(clf_path,  allow_pickle=True)['d'].item()
clft_f = np.load(clft_path, allow_pickle=True)['d'].item() if os.path.exists(clft_path) else None
print(f'clf_R: {len(clf_f):,} cached pairs | clf_topic: {len(clft_f):,} pairs' if clft_f else f'clf_R: {len(clf_f):,} | clf_topic: NOT CACHED')

## Re-fit multi-view LambdaMART (same as train_ensemble_full, seconds)

In [ ]:
FEATURES = ['bm25', 'bm25_rank', 'dense', 'dense_rank', 'rrf', 'clf_rel', 'clf_partial']
if clft_f is not None: FEATURES += ['clf_topic_rel', 'clf_topic_partial']
FEATURES += ['llm_yesno']
if topi: FEATURES += ['topicality']
if cm:   FEATURES += ['condition_match']
print('FEATURES:', FEATURES)

def featvec(s, t, d):
    rf = rfeat.get((s, t, d), {}); cr, cp = clf_f.get((s, t, d), (0., 0.))
    v = {'bm25': rf.get('bm25', 0.), 'bm25_rank': rf.get('bm25_rank', cfg.cand_k),
         'dense': rf.get('dense', 0.), 'dense_rank': rf.get('dense_rank', cfg.cand_k),
         'rrf': rf.get('rrf', 0.), 'clf_rel': cr, 'clf_partial': cp,
         'llm_yesno': llm.get((s, t, d), cfg.llm_floor)}
    if clft_f is not None:
        tr, tp = clft_f.get((s, t, d), (0., 0.))
        v['clf_topic_rel'] = tr; v['clf_topic_partial'] = tp
    if topi: v['topicality'] = topi.get((s, t, d), 0.)
    if cm:   v['condition_match'] = cm.get((s, t, d), 0.)
    return [v[f] for f in FEATURES]

def build(s):
    X, y, g = [], [], []
    rel = sets[s]['rel_dict']
    for t, docs in pool[s].items():
        docs = [d for d in docs if d in id2fields]; g.append(len(docs))
        for d in docs: X.append(featvec(s, t, d)); y.append(int(rel[t].get(d, 0)))
    return np.array(X, dtype=np.float32), np.array(y), g

X21, y21, g21 = build('trec21'); Xkz, ykz, gkz = build('kz')
Xte, yte, gte = build('trec22')
Xtr = np.vstack([X21, Xkz]); ytr = np.concatenate([y21, ykz]); gtr = g21 + gkz
print('train', Xtr.shape, '| test', Xte.shape)

In [ ]:
# Backward selection + num_leaves on TREC21 CV (same as train_ensemble_full).
topic_of = np.concatenate([[i]*c for i,c in enumerate(g21)]); nt = len(g21)
fold = np.random.default_rng(cfg.seed).integers(0, 5, nt)

def _ndcg(y, s):
    o = np.argsort(-s)[:10]; gg = (2.0**y[o]-1)
    d = 1/np.log2(np.arange(2, 2+len(o)))
    idcg = ((2.0**np.sort(y)[::-1][:10]-1)/np.log2(np.arange(2, 2+min(10, len(y))))).sum()
    return (gg*d).sum()/idcg if idcg > 0 else 0.0

def cv(cols, nl):
    vals = []
    for f in range(5):
        trm = np.isin(topic_of, [i for i in range(nt) if fold[i] != f]); vm = ~trm
        gt = [g21[i] for i in range(nt) if fold[i] != f]
        gv = [g21[i] for i in range(nt) if fold[i] == f]
        b = lgb.train(
            {'objective':'lambdarank','metric':'ndcg','ndcg_eval_at':[10],
             'num_leaves':nl,'min_data_in_leaf':20,'learning_rate':0.05,'lambda_l2':1.0,'verbose':-1},
            lgb.Dataset(X21[trm][:,cols], y21[trm], group=gt), num_boost_round=50)
        p = b.predict(X21[vm][:,cols]); i0 = 0
        for gg in gv: vals.append(_ndcg(y21[vm][i0:i0+gg], p[i0:i0+gg])); i0 += gg
    return float(np.mean(vals))

cols = list(range(len(FEATURES))); cur = cv(cols, 15)
while len(cols) > 1:
    cand = [(cv([c for c in cols if c != j], 15), j) for j in cols]; bv, bj = max(cand)
    if bv > cur + 1e-4: cur = bv; cols.remove(bj); print('drop', FEATURES[bj], '-> CV', round(bv, 4))
    else: break
NL = max([7, 15, 31], key=lambda nl: cv(cols, nl))
SEL = cols
print('SELECTED:', [FEATURES[c] for c in SEL], '| num_leaves', NL, '| trec21 CV', round(cur, 4))

In [ ]:
# Train on full train set; get TREC22 predictions.
booster = lgb.train(
    {'objective':'lambdarank','metric':'ndcg','ndcg_eval_at':[10],
     'num_leaves':NL,'min_data_in_leaf':20,'learning_rate':0.05,'lambda_l2':1.0,'verbose':-1},
    lgb.Dataset(Xtr[:,SEL], ytr, group=gtr), num_boost_round=50)

pred = booster.predict(Xte[:, SEL])

# Build per-doc records with (topic, doc, label, rank, features) for analysis below.
records, i0 = [], 0
for t in pool['trec22']:
    docs = [d for d in pool['trec22'][t] if d in id2fields]
    n = len(docs)
    scores = pred[i0:i0+n]; labels = yte[i0:i0+n]; feats = Xte[i0:i0+n]
    rank_order = np.argsort(-scores)
    ranks = np.empty(n, dtype=int); ranks[rank_order] = np.arange(1, n+1)
    for j, d in enumerate(docs):
        rec = {'topic_id': t, 'doc_id': d,
               'label': int(labels[j]), 'rank': int(ranks[j]), 'score': float(scores[j])}
        for ci, c in enumerate(SEL):
            rec[FEATURES[c]] = float(feats[j, ci])
        records.append(rec)
    i0 += n

# Quick sanity: confirm TREC22 NDCG@10.
run = {r['topic_id']: {} for r in records}
for r in records: run[r['topic_id']][r['doc_id']] = r['score']
qrels = {t: {d: int(v) for d,v in sets['trec22']['rel_dict'][t].items()} for t in run}
m = pytrec_metrics(run, qrels, k=10)
print(f"TREC22: NDCG@10={m['ndcg@10']}  P@10={m['P@10']}  MRR={m['mrr']}")
print('importances:', dict(zip([FEATURES[c] for c in SEL], booster.feature_importance().tolist())))

## 1 — Feature profile by outcome: what separates surfaced from buried eligibles?

In [ ]:
surfaced_r2 = [r for r in records if r['label'] == 2 and r['rank'] <= 10]
buried_r2   = [r for r in records if r['label'] == 2 and r['rank'] >  10]
fp_top10    = [r for r in records if r['label'] == 0 and r['rank'] <= 10]
print(f'surfaced_r2={len(surfaced_r2):,}  buried_r2={len(buried_r2):,}  FP_top10={len(fp_top10):,}')

feat_keys = [FEATURES[c] for c in SEL]

def fmeans(bucket):
    return {k: np.nanmean([r.get(k, np.nan) for r in bucket]) for k in feat_keys}

s2, b2, fp = fmeans(surfaced_r2), fmeans(buried_r2), fmeans(fp_top10)

rows = []
for k in feat_keys:
    gap_pct = abs(s2[k] - b2[k]) / max(abs(s2[k]), abs(b2[k]), 1e-9) * 100
    rows.append({'feature': k, 'surfaced_r2': round(s2[k],4),
                 'buried_r2': round(b2[k],4), 'fp_top10': round(fp[k],4),
                 's2_vs_b2_%': round(gap_pct,1)})

df_feat = pd.DataFrame(rows).sort_values('s2_vs_b2_%', ascending=False)
print('Feature means by bucket (sorted by surfaced-vs-buried gap):')
df_feat

## 2 — Displacer analysis: which feature makes FPs beat buried eligibles?

In [ ]:
from collections import defaultdict, Counter

by_topic = defaultdict(list)
for r in records: by_topic[r['topic_id']].append(r)

# For each buried_r2, collect the top-10 non-r2 docs that outranked it.
displacers = []
for r in buried_r2:
    top10 = [d for d in by_topic[r['topic_id']] if d['rank'] <= 10 and d['label'] < 2]
    displacers.extend(top10)

dp = fmeans(displacers)
print(f'Displacers: n={len(displacers):,}  label mix: {dict(Counter(d["label"] for d in displacers))}')

rows = []
for k in feat_keys:
    diff = dp[k] - b2[k]
    rows.append({'feature': k, 'buried_r2': round(b2[k],4), 'displacer': round(dp[k],4),
                 'delta (d-b)': round(diff,4),
                 'who wins': 'DISPLACER' if diff > 0.02 else ('buried' if diff < -0.02 else '≈ tied')})

df_disp = pd.DataFrame(rows).sort_values('delta (d-b)', ascending=False)
print('\nFeature comparison: buried_r2 vs what outranked them (displacer):')
df_disp

## 3 — Eligibility retention: does clf_long add anything elig_first-L512 can't already see?

For 300 sampled buried_r2 docs, computes the fraction of their eligibility tokens that survive
into the `elig_first-L512` model input (the same text clf_R is trained and scored on).

- **If ≥ 0.90 mean → elig_first-L512 already covers the eligibility text.** clf_long can't
  add signal that isn't already in clf_R's input. The burial is discriminative capacity
  or calibration, not context length. → Favour **monoT5** (architecture + scale diversity).
- **If < 0.70 mean → substantial eligibility text IS being cut off.** clf_long has a real
  mechanism on this subset. → **clf_long** is justified (but note Longformer needs
  `global_attention_mask` — not a drop-in swap into retrain_classifier).
- **Exclusion < 0.50 for many docs** even when inclusion is retained → exclusion criteria are
  pushed off by long inclusion lists. `budget_incexc` (already in experiments.py, zero retrain)
  is the cheap probe before a 4096-token training run.

In [ ]:
from transformers import AutoTokenizer
from ctmatch.experiments import resolve_ckpt

# Use the clf_R tokenizer — token counts must match the model that actually truncates these docs.
# resolve_ckpt returns the local Drive path if present, otherwise the HF hub id.
print('Loading clf_R tokenizer (no GPU)...')
tokenizer = AutoTokenizer.from_pretrained(resolve_ckpt(cfg, cfg.clf_ckpt), use_fast=True)

# Sample 300 buried_r2 docs (or all if fewer).
SAMPLE = buried_r2[:300]
# Generic topic — retention depends mostly on the doc (eligibility length), not the specific topic.
GENERIC_TOPIC = '65-year-old male with type 2 diabetes mellitus and hypertension'

elig_fracs, excl_fracs = [], []
for r in SAMPLE:
    flds = id2fields.get(r['doc_id'], {})
    elig_fracs.append(eligibility_retained_frac(tokenizer, GENERIC_TOPIC, flds, cfg))
    excl_fracs.append(exclusion_retained_frac(tokenizer, GENERIC_TOPIC, flds, cfg))

ef, xf = np.array(elig_fracs), np.array(excl_fracs)
print(f'\nn={len(SAMPLE)} buried_r2 docs sampled')
print(f'Eligibility retained @ elig_first-L512:')
print(f'  mean={ef.mean():.3f}  median={np.median(ef):.3f}  '
      f'pct<0.50={np.mean(ef<0.50)*100:.1f}%  pct<0.70={np.mean(ef<0.70)*100:.1f}%  '
      f'pct<0.90={np.mean(ef<0.90)*100:.1f}%')
print(f'Exclusion  retained @ elig_first-L512:')
print(f'  mean={xf.mean():.3f}  median={np.median(xf):.3f}  '
      f'pct<0.50={np.mean(xf<0.50)*100:.1f}%  pct<0.70={np.mean(xf<0.70)*100:.1f}%  '
      f'pct<0.90={np.mean(xf<0.90)*100:.1f}%')

In [ ]:
# Histogram of eligibility retention fracs.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, fracs, title in zip(axes, [elig_fracs, excl_fracs],
                             ['Eligibility retention (elig_first-L512)',
                              'Exclusion retention (elig_first-L512)']):
    ax.hist(fracs, bins=20, color='steelblue', edgecolor='white')
    ax.axvline(np.mean(fracs), color='red', linestyle='--', label=f'mean={np.mean(fracs):.2f}')
    ax.set_title(title + f'\n(n={len(fracs)} buried_r2 docs)')
    ax.set_xlabel('Fraction retained'); ax.legend()
plt.tight_layout(); plt.show()

## 4 — Decision guide

Read the three outputs above to pick the next lever.

### From the DISPLACER TABLE (cell 2):
What feature does 'DISPLACER wins' appear next to?

| Winning feature | Diagnosis | Next lever |
|---|---|---|
| `clf_rel` / `clf_topic_rel` | CE mis-calibrated (scores FPs high and buried eligibles low) | More CE capacity: `clf_long` OR monoT5 architecture |
| `topicality` / `condition_match` | Topicality view outweighs eligibility; buried docs match topic but lose on topicality | Better topicality model (MedCPT bi-encoder, SapBERT++) |
| `rrf` / `bm25` / `dense` | Retrieval rank wins over reranker scores | Ensemble re-weighting; check if retrieval features are over-weighted |
| `llm_yesno` | LLM judge is still driving (even if dropped, check it survived selection) | Wider LLM coverage or judge fine-tune |

### From the ELIGIBILITY RETENTION (cell 3):

| Result | Interpretation | Build |
|---|---|---|
| eligibility mean ≥ 0.90 | elig_first-L512 already covers the text; clf_long can't add new signal | **monoT5-3B** (architecture + scale diversity) |
| exclusion pct<0.50 > 20% | Exclusion pushed off by long inclusion | `budget_incexc` repr (zero retrain) as cheap probe, then `clf_long` |
| eligibility mean < 0.70 | Meaningful eligibility truncation on buried docs | **clf_long** (but use `allenai/longformer-base-4096` + global_attention_mask fix) |

### Reminder: gate on TREC21 CV, not TREC22
The 0.5658 is the TREC22 test number and has been touched many times.
Any new view bake-off belongs on **TREC21 CV** as the primary gate;
TREC22 is touched once only after a clear TREC21 gain.